# Robustness check: processed (lemma) vs. raw exact suffixes

**Purpose**: Compare quarterly relative shares of the three target terms using
`context_processed` (lemma-reduced) against `context` (raw exact suffixes).

**Period**: 2022-04-21 to 2025-02-08 (scraper cutoff to last crawled date; update 2026-03-20)

**Target terms**: Klimawandel, Klimakrise, Klimaschutz

**Setup in this notebook**: load, compare, plot (single figure with 6 lines).

## Setup: imports and data loading

In [ ]:
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

notebook_dir = (
    os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else os.getcwd()
)
sys.path.append(os.path.join(notebook_dir, "..", "pylib"))

from config import DEFAULT_END_DATE, DWH_DB_PATH, PLOTS_DIR, SCRAPER_CUTOFF
from handle_sqlite import read_table_as_dataframe
from plotting_utils import apply_plot_style

apply_plot_style()
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
plot_dir = PLOTS_DIR

analysis_start = SCRAPER_CUTOFF
analysis_end = DEFAULT_END_DATE

# Load raw and processed context plus metadata for timestamps
df_context_raw = read_table_as_dataframe("context", str(DWH_DB_PATH))
df_context_processed = read_table_as_dataframe("context_processed", str(DWH_DB_PATH))
df_meta = read_table_as_dataframe("newspapers", str(DWH_DB_PATH))
df_meta["data_published"] = pd.to_datetime(df_meta["data_published"])

df_meta_filtered = df_meta[df_meta["data_published"] >= analysis_start].copy()
if analysis_end is not None:
    df_meta_filtered = df_meta_filtered[df_meta_filtered["data_published"] <= analysis_end].copy()

# Attach publication date to context rows
raw = df_context_raw.merge(
    df_meta_filtered[["newspaper_id", "data_published"]],
    on="newspaper_id",
    how="inner",
)
processed = df_context_processed.merge(
    df_meta_filtered[["newspaper_id", "data_published"]],
    on="newspaper_id",
    how="inner",
)

RAW_SUFFIX_MAP = {
    "wandel": "Klimawandel",
    "krise": "Klimakrise",
    "schutz": "Klimaschutz",
}
PROCESSED_SUFFIX_MAP = {
    "wandel": "Klimawandel",
    "krise": "Klimakrise",
    "schutz": "Klimaschutz",
}

raw["term"] = raw["suffix"].str.lower().str.strip().map(RAW_SUFFIX_MAP)
processed["term"] = processed["suffix_lemma"].str.lower().str.strip().map(PROCESSED_SUFFIX_MAP)

df_raw = raw[raw["term"].notna()].copy()
df_processed = processed[processed["term"].notna()].copy()

print(f"Raw mentions (exact suffixes): {len(df_raw)}")
print(f"Processed mentions (lemma): {len(df_processed)}")
print(f"Raw period: {df_raw['data_published'].min()} to {df_raw['data_published'].max()}")
print(f"Processed period: {df_processed['data_published'].min()} to {df_processed['data_published'].max()}")

## Figure: quarterly shares, processed vs. raw (6-line comparison)

In [ ]:
terms_plot = ["Klimaschutz", "Klimawandel", "Klimakrise"]
linestyles = {"Klimaschutz": "-", "Klimawandel": "--", "Klimakrise": ":"}
markers = {"Klimaschutz": "o", "Klimawandel": "s", "Klimakrise": "^"}

processed_q = df_processed.copy()
processed_q["quarter"] = processed_q["data_published"].dt.to_period("Q")
processed_quarterly = processed_q.groupby(["quarter", "term"]).size().unstack(fill_value=0)

raw_q = df_raw.copy()
raw_q["quarter"] = raw_q["data_published"].dt.to_period("Q")
raw_quarterly = raw_q.groupby(["quarter", "term"]).size().unstack(fill_value=0)

quarter_index = processed_quarterly.index.union(raw_quarterly.index)
processed_quarterly = processed_quarterly.reindex(index=quarter_index, columns=terms_plot, fill_value=0)
raw_quarterly = raw_quarterly.reindex(index=quarter_index, columns=terms_plot, fill_value=0)

processed_shares = (processed_quarterly.div(processed_quarterly.sum(axis=1), axis=0) * 100).fillna(0)
raw_shares = (raw_quarterly.div(raw_quarterly.sum(axis=1), axis=0) * 100).fillna(0)

x = np.arange(len(quarter_index))
fig, ax = plt.subplots(figsize=(12, 6))

for term in terms_plot:
    ax.plot(
        x,
        processed_shares[term].values,
        linestyle=linestyles[term],
        marker=markers[term],
        color="black",
        linewidth=1.8,
        markersize=5,
        label=f"{term} (processed)",
    )
    ax.plot(
        x,
        raw_shares[term].values,
        linestyle=linestyles[term],
        marker="x",
        color="dimgray",
        linewidth=1.2,
        markersize=5,
        alpha=0.9,
        label=f"{term} (raw)",
    )

ax.set_xticks(x)
ax.set_xticklabels([str(q) for q in quarter_index], rotation=45, ha="right")
ax.set_xlabel("Quarter")
ax.set_ylabel("Relative share (%)")
ax.set_title("Processed (lemma) vs. raw exact suffixes: quarterly shares")
ax.grid(True, axis="y", alpha=0.3)
ax.legend(loc="best", ncol=2, frameon=True)

plt.tight_layout()
plot_path = os.path.join(plot_dir, "07_abweichungen_reduktion_vs_roh.png")
plt.savefig(plot_path, dpi=300, bbox_inches="tight", facecolor="white")
print(f"Figure saved: {plot_path}")
plt.show()